# 实验六：回溯与分支限界算法

---

In [1]:
#include <stdbool.h>
#include <stdio.h>

## 题目1：最大团问题

**实验目的**

强化学生利用回溯算法和优化处理实际问题的能力。

**实验内容**

给定无向图 $G=(V,E)$，如果一个顶点集合中任意两个不同顶点之间都有边，它就构成完全子图。若它无法通过加入其他顶点继续扩大，则称为极大团。最大团是顶点数最多的团，本题求这个顶点数。

**输入与输出**

输入为顶点数 $n$ 和无向图的邻接矩阵。顶点编号为 $0$ 到 $n-1$，矩阵对称，有边记为 $1$，无边记为 $0$，主对角线填 $0$。输出最大团的顶点数，空图的结果为 $0$。

### 算法思路

**搜索状态与分支**

按编号依次决定是否选入每个顶点。状态 `(vertex, size)` 表示下一步要处理顶点 `vertex`，而 `chosen[0..size)` 是由编号小于 `vertex` 的已选顶点构成的团。每次进入搜索，用当前团的大小更新 `best`。当 `vertex == n` 时，所有顶点都已决定。

- **选入。** 顶点 `vertex` 必须与每个已选顶点相邻。检查通过后，写入 `chosen[size]`，递归处理 `(vertex + 1, size + 1)`。
- **跳过。** 递归处理 `(vertex + 1, size)`。这一分支也会在选入分支返回后执行。每层的 `size` 按值传递，沿用原值就能恢复当前团的有效前缀。

例如，图中有边 `0–1`、`0–2`、`1–2`、`0–3`。已选 `{0,1}` 时，`adj[2][0]` 和 `adj[2][1]` 都是 1，因此可以选入 `2`。虽然 `adj[3][0] = 1`，但 `adj[3][1] = 0`，所以 `3` 无法选入。当前团为空时，没有任何已选顶点需要检查。

**手推选入与跳过**

考虑只有边 `0–1`、`2–3`、`2–4`、`3–4` 的图。用 `(下一个顶点, 已选集合)` 表示递归状态。

| 搜索路径 | 递归状态 | 结果 |
| --- | --- | --- |
| 先选入 0 和 1 | `(0, ∅) → (1, {0}) → (2, {0,1})` | 找到大小为 2 的团，顶点 2、3、4 都无法加入 |
| 回到根节点并跳过 0 | `(0, ∅) → (1, ∅)` | 继续考虑顶点 1 的选入与跳过 |
| 跳过 0 和 1，依次选入 2、3、4 | `(2, ∅) → (3, {2}) → (4, {2,3}) → (5, {2,3,4})` | 找到大小为 3 的最大团 |

到达 `(5, {2,3,4})` 时已处理完所有顶点。先更新 `best` 再判断递归终点，才能将最后一次选入得到的团计入答案。

**规模上界与正确性**

待决定的顶点共有 `n - vertex` 个。即使它们全部可以选入，当前分支也最多得到 `size + (n - vertex)` 个顶点。当这个上界小于或等于 `best` 时，题目所求的最大团大小无法继续提高，可以结束当前分支。对于上面的图，找到 `{0,1}` 后，沿选入 0、跳过 1、2、3 的路径到达 `(4, {0})`。此时 `best = 2`，上界为 `1 + (5 - 4) = 2`，因此可以直接返回。

搜索从空团开始。跳过顶点保留原来的团，选入顶点通过相容性检查后仍得到团，所以 `best` 总是一个已找到的可行大小。对于任意团，按照编号选入该团中的顶点、跳过其余顶点，选入时都会通过相容性检查。基础回溯因此覆盖所有团。上界只会略过无法提高已知最优值的子树，故剪枝后的 `best` 仍是最大团大小。

### 伪代码

```text
MAXIMUM-CLIQUE(adj, n)
    chosen ← 长度为 n 的工作数组
    best ← 0
    定义 SEARCH(v, size)                   // 共用 adj、n、chosen 和 best
        best ← max(best, size)        // 当前已选顶点已构成团
        若 v = n，返回               // 所有顶点都已决定
        upper ← size + (n - v)        // 假定剩余顶点全部能选入
        若 upper ≤ best，返回

        若 CAN-JOIN(adj, chosen, size, v)
            chosen[size] ← v
            SEARCH(v + 1, size + 1)   // 选入 v
        SEARCH(v + 1, size)           // 跳过 v

    SEARCH(0, 0)
    返回 best

CAN-JOIN(adj, chosen, size, v)
    对于 i 从 0 到 size - 1
        若 adj[v][chosen[i]] = 0，返回 false
    返回 true                         // 所有已选顶点都与 v 相邻
```

### 源代码实现

C23 实现将最大顶点数设为 `MAX_VERTICES = 32`，要求 `0 <= n <= MAX_VERTICES`。有效邻接矩阵存放在固定容量数组的左上角 `n × n` 区域。入口函数 `maximum_clique_size(adj, n)` 返回最大团大小。

In [2]:
enum { MAX_VERTICES = 32 };

static bool can_join(const int adj[][MAX_VERTICES], const int chosen[],
                     int size, int vertex)
{
    // 候选顶点必须与有效前缀中的每个顶点相邻。
    for (int i = 0; i < size; i++) {
        if (adj[vertex][chosen[i]] == 0) {
            return false;
        }
    }
    return true;
}

In [3]:
static void search_clique(const int adj[][MAX_VERTICES], int n, int vertex,
                          int chosen[], int size, int *best)
{
    if (size > *best) {
        *best = size;
    }
    if (vertex == n) {
        return;
    }

    // 即使选入全部剩余顶点，也无法得到更大的团。
    if (size + (n - vertex) <= *best) {
        return;
    }

    if (can_join(adj, chosen, size, vertex)) {
        chosen[size] = vertex;
        search_clique(adj, n, vertex + 1, chosen, size + 1, best);
    }

    // 跳过当前顶点，继续使用选入前的有效前缀。
    search_clique(adj, n, vertex + 1, chosen, size, best);
}

int maximum_clique_size(const int adj[][MAX_VERTICES], int n)
{
    int chosen[MAX_VERTICES];
    int best = 0;

    search_clique(adj, n, 0, chosen, 0, &best);
    return best;
}

### 测试用例

从头运行笔记本，先执行头文件和函数定义单元。相容性检查覆盖候选顶点与所有已选顶点相邻、缺少一条边和当前团为空三种情况。随后五组用例分别检查空图、无边图、四边形、两个分量中存在不同大小的团、完全图。每组输出手算期望值和实际值。

In [4]:
{
    const int adj[MAX_VERTICES][MAX_VERTICES] = {
        {0, 1, 1, 1},
        {1, 0, 1, 0},
        {1, 1, 0, 0},
        {1, 0, 0, 0}
    };
    const int chosen[] = {0, 1};

    printf("检查 2 能否加入 {0,1}：%d（预期 1）\n",
           can_join(adj, chosen, 2, 2));
    printf("检查 3 能否加入 {0,1}：%d（预期 0）\n",
           can_join(adj, chosen, 2, 3));
    printf("检查 0 能否加入空集合：%d（预期 1）\n",
           can_join(adj, chosen, 0, 0));
}

检查 2 能否加入 {0,1}：1（预期 1）
检查 3 能否加入 {0,1}：0（预期 0）
检查 0 能否加入空集合：1（预期 1）


In [5]:
static void check_clique_case(const char *name, const int adj[][MAX_VERTICES],
                              int n, int expected)
{
    int actual = maximum_clique_size(adj, n);

    printf("%s：实际 %d，预期 %d，%s\n",
           name, actual, expected, actual == expected ? "通过" : "失败");
}

In [6]:
{
    const int adj[MAX_VERTICES][MAX_VERTICES] = {{0}};
    check_clique_case("空图", adj, 0, 0);
}

空图：实际 0，预期 0，通过


In [7]:
{
    const int adj[MAX_VERTICES][MAX_VERTICES] = {{0}};
    check_clique_case("无边图", adj, 3, 1);
}

无边图：实际 1，预期 1，通过


In [8]:
{
    const int adj[MAX_VERTICES][MAX_VERTICES] = {
        {0, 1, 0, 1},
        {1, 0, 1, 0},
        {0, 1, 0, 1},
        {1, 0, 1, 0}
    };
    check_clique_case("四边形", adj, 4, 2);
}

四边形：实际 2，预期 2，通过


In [9]:
{
    const int adj[MAX_VERTICES][MAX_VERTICES] = {
        {0, 1, 0, 0, 0},
        {1, 0, 0, 0, 0},
        {0, 0, 0, 1, 1},
        {0, 0, 1, 0, 1},
        {0, 0, 1, 1, 0}
    };
    check_clique_case("一条边和一个三角形", adj, 5, 3);
}

一条边和一个三角形：实际 3，预期 3，通过


In [10]:
{
    const int adj[MAX_VERTICES][MAX_VERTICES] = {
        {0, 1, 1, 1},
        {1, 0, 1, 1},
        {1, 1, 0, 1},
        {1, 1, 1, 0}
    };
    check_clique_case("完全图", adj, 4, 4);
}

完全图：实际 4，预期 4，通过


#### 剪枝前后对比

同一份 C23 测试代码和五组输入，分别运行基础回溯和加上界的版本。每次进入搜索函数计一次，包含根节点、递归终点和立即返回的剪枝节点。

| 用例 | 基础回溯结果 | 剪枝后结果 | 基础回溯调用数 | 剪枝后调用数 |
| --- | ---: | ---: | ---: | ---: |
| 空图 | 0 | 0 | 1 | 1 |
| 无边图 | 1 | 1 | 10 | 8 |
| 四边形 | 2 | 2 | 22 | 13 |
| 一条边和一个三角形 | 3 | 3 | 30 | 20 |
| 完全图 | 4 | 4 | 31 | 9 |

空图从根节点直接到达递归终点。完全图先沿全部选入的路径找到大小为 4 的团，随后每个跳过一个顶点的分支上界都只有 3。

复测调用数时，每组测试先将局部计数清零，把它的地址传入搜索函数，并在函数入口累加。保持顶点顺序和相容性检查相同，分别启用和去掉规模上界判断。

### 复杂度分析

以顶点数 $n$ 为输入规模，分析邻接矩阵已经建好后，一次求解最大团大小所需的时间和空间。

#### 时间复杂度

**每层最多进入多少次搜索？** 根节点深度为 $0$。每次递归都令 `vertex` 增加 $1$，所以深度 $d$ 表示已经决定了前 $d$ 个顶点。每个节点最多进入选入和跳过两个分支，因此第 $d$ 层最多有 $2^d$ 次调用。处理完所有顶点时到达深度 $n$。

**一次调用最多做多少工作？** 深度为 $d$ 时，已选顶点数 `size` 至多为 $d$。邻接矩阵允许用 $O(1)$ 时间查询一条边，`can_join` 最多查询 $d$ 次。更新最优值、判断递归终点、检查上界和准备递归参数各用常数时间。子调用在各自所在层计算，所以当前调用的工作量上界为 $O(d+1)$。深度 $n$ 的终止节点只需常数时间，也满足这个上界。

把每层的调用数上界乘以单次调用的工作量上界，再对各层求和。忽略常数因子，得到

$$
\sum_{d=0}^{n} 2^d(d+1)
\;\le\; (n+1)\sum_{d=0}^{n} 2^d
\;=\; (n+1)(2^{n+1}-1).
$$

因此基础回溯在 $n\ge 1$ 时的最坏时间复杂度上界是 $O(n2^n)$。空图只进入一次递归，耗时为常数。

#### 为什么加了剪枝还能沿用这个上界？

当 `size + (n - vertex) <= *best` 时，当前调用直接返回，其下方的子节点都不会进入。对于同一输入，剪枝版只会访问基础回溯搜索树的一部分，每层仍至多有 $2^d$ 次调用。每次进入时新增的上界判断只用常数时间，所以单次调用仍至多花费 $O(d+1)$。代入同一个求和式，剪枝版也有 $O(n2^n)$ 的最坏时间上界。

例如，4 个顶点的完全图在实测中从 31 次调用降到 9 次。这个上界覆盖各种输入，实际减少的调用数取决于多早找到较大的团，以及有多少状态的上界能被当前最优值压住。

#### 空间复杂度

- **输入图。** 按问题规模扩展时，$n\times n$ 的邻接矩阵占用 $\Theta(n^2)$ 空间。递归各层共用它。
- **已选顶点。** `chosen` 在入口只创建一次。按 $n$ 分配时需要 $\Theta(n)$ 空间，每层用自己的 `size` 记录有效前缀长度。
- **递归栈。** 从根节点到终止节点最多同时保留 $n+1$ 层，每层只保存常数个参数和局部变量，因此栈空间为 $O(n)$（$n\ge 1$）。

按 $n$ 扩展时，求解的额外空间为 $O(n)$，计入输入矩阵的总空间为 $\Theta(n^2)$。当前 C23 单元用固定容量 `MAX_VERTICES = 32`，每个图的矩阵实际预留 $32\times32$ 个 `int`，工作数组预留 32 个 `int`。递归所需栈空间仍取决于本次输入的 $n$。

### 实验总结

在最大团搜索中，选入分支靠相容性检查维持团的性质，跳过分支保留寻找其他团的机会。两条分支通过不同的 `size` 使用同一数组的有效前缀，因此回溯时只需传递正确的长度。以「一条边和一个三角形」为例，先选入顶点 0 得到大小为 2 的极大团，继续搜索跳过 0 的分支才找到大小为 3 的最大团。

基于剩余顶点数的上界保留了五组用例的正确结果。四组用例的递归调用数下降，其中完全图从 31 次降到 9 次。剪枝能省去多少搜索取决于图结构、顶点顺序和当前已找到的最优值。